# Lab 14-15 · Rooflines and benchmark means

This notebook runs the three parts of the lab with `roofline.py`. **Predict first** at each step, then run the cell and compare.

In [1]:
import math
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import roofline as rf

for name, (peak, bw) in rf.MACHINES.items():
    print(f"{name:22s} peak {peak / 1e12:8.1f} TFLOP/s  bandwidth {bw / 1e12:5.2f} TB/s  ridge {rf.ridge(peak, bw):6.1f} FLOP/byte")

H100 SXM, FP32         peak     67.0 TFLOP/s  bandwidth  3.35 TB/s  ridge   20.0 FLOP/byte
H100 SXM, BF16 dense   peak    989.5 TFLOP/s  bandwidth  3.35 TB/s  ridge  295.4 FLOP/byte
TPU v4, BF16           peak    275.0 TFLOP/s  bandwidth  1.20 TB/s  ridge  229.2 FLOP/byte
TPU7x, BF16            peak   2307.0 TFLOP/s  bandwidth  7.38 TB/s  ridge  312.6 FLOP/byte


## Part 1 · Rooflines

*Predict:* which machine is fastest for LLM decoding at 1 FLOP/byte? (The one with the most memory bandwidth, whatever its peak.)

In [2]:
ai = np.logspace(-2, 4, 400)
fig, ax = plt.subplots(figsize=(7, 4.5))
for name, (peak, bw) in rf.MACHINES.items():
    ax.loglog(ai, [rf.attainable(peak, bw, x) / 1e12 for x in ai], label=name)
for kname, (flop, nbytes) in rf.KERNELS.items():
    ax.axvline(rf.intensity(flop, nbytes), color="gray", lw=0.6, ls=":")
ax.set_xlabel("Arithmetic intensity (FLOP/byte)")
ax.set_ylabel("Attainable TFLOP/s")
ax.legend(fontsize=8)
ax.grid(True, which="both", alpha=0.3)
fig.tight_layout()

best = max(rf.MACHINES, key=lambda m: rf.attainable(*rf.MACHINES[m], 1.0))
print("fastest at 1 FLOP/byte:", best)

fastest at 1 FLOP/byte: TPU7x, BF16


## Part 2 · Bottlenecks

For each kernel and machine: attainable TFLOP/s, the fraction of peak, and the intensity needed to reach the roof (the ridge).

In [3]:
for kname, (flop, nbytes) in rf.KERNELS.items():
    x = rf.intensity(flop, nbytes)
    print(f"{kname} (intensity {x:.3f} FLOP/byte)")
    for name, (peak, bw) in rf.MACHINES.items():
        att = rf.attainable(peak, bw, x)
        print(f"    {name:22s} {att / 1e12:8.3f} TFLOP/s  {100 * att / peak:6.2f}% of peak  needs {rf.ridge(peak, bw):6.1f}")
print("GEMM, n = 4096, FP32:", round(rf.gemm_intensity(4096, 4), 1), "FLOP/byte")

vector add z = x + y (intensity 0.083 FLOP/byte)
    H100 SXM, FP32            0.279 TFLOP/s    0.42% of peak  needs   20.0
    H100 SXM, BF16 dense      0.279 TFLOP/s    0.03% of peak  needs  295.4
    TPU v4, BF16              0.100 TFLOP/s    0.04% of peak  needs  229.2
    TPU7x, BF16               0.615 TFLOP/s    0.03% of peak  needs  312.6
SAXPY y = a x + y (intensity 0.167 FLOP/byte)
    H100 SXM, FP32            0.558 TFLOP/s    0.83% of peak  needs   20.0
    H100 SXM, BF16 dense      0.558 TFLOP/s    0.06% of peak  needs  295.4
    TPU v4, BF16              0.200 TFLOP/s    0.07% of peak  needs  229.2
    TPU7x, BF16               1.230 TFLOP/s    0.05% of peak  needs  312.6
7-point stencil (intensity 1.000 FLOP/byte)
    H100 SXM, FP32            3.350 TFLOP/s    5.00% of peak  needs   20.0
    H100 SXM, BF16 dense      3.350 TFLOP/s    0.34% of peak  needs  295.4
    TPU v4, BF16              1.200 TFLOP/s    0.44% of peak  needs  229.2
    TPU7x, BF16               7.380 

## Part 3 · Which mean?

Three machines' times (seconds) on five programs. Normalize to each machine in turn and compare the means. *Predict:* which mean gives the same ranking whatever the reference?

In [4]:
times = {"A": [10.0, 20.0, 40.0, 5.0, 80.0], "B": [12.0, 15.0, 30.0, 8.0, 60.0], "C": [8.0, 25.0, 50.0, 4.0, 100.0]}
for ref in times:
    print(f"reference {ref}:")
    for m, t in times.items():
        ratios = [r / x for r, x in zip(times[ref], t)]
        print(f"    {m}: amean {rf.amean(ratios):.3f}  gmean {rf.gmean(ratios):.3f}  hmean {rf.hmean(ratios):.3f}")
rank = {ref: sorted(times, key=lambda m: -rf.gmean([r / x for r, x in zip(times[ref], times[m])])) for ref in times}
print("geometric-mean ranking by reference:", rank)
assert len({tuple(v) for v in rank.values()}) == 1, "the geometric mean's ranking should not depend on the reference"

reference A:
    A: amean 1.000  gmean 1.000  hmean 1.000
    B: amean 1.092  gmean 1.043  hmean 0.990
    C: amean 0.980  gmean 0.956  hmean 0.935
reference B:
    A: amean 1.010  gmean 0.959  hmean 0.916
    B: amean 1.000  gmean 1.000  hmean 1.000
    C: amean 1.060  gmean 0.917  hmean 0.811
reference C:
    A: amean 1.070  gmean 1.046  hmean 1.020
    B: amean 1.233  gmean 1.091  hmean 0.943
    C: amean 1.000  gmean 1.000  hmean 1.000
geometric-mean ranking by reference: {'A': ['B', 'A', 'C'], 'B': ['B', 'A', 'C'], 'C': ['B', 'A', 'C']}


**Discussion.** The geometric mean of ratios gives the same ranking whatever machine is the reference, which is why SPEC uses it. The arithmetic and harmonic means of ratios can flip with the reference. When the quantity is a rate over equal work, the harmonic mean of rates gives the true average rate (Chapter 14-15, §3.3).